# Lab 24-03 · CTLE, FFE, DFE and CDR

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import serdes


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · CTLE

*Predict* the DC gain, the peak gain and its frequency, and the gain at 16 GHz.

In [2]:
gm, rd, rs, cl = 20e-3, 200.0, 400.0, 40e-15
cs = 1 / (2 * np.pi * rs * 2e9)
f = np.logspace(8, 11, 2000)
with_pole = serdes.ctle(f, gm, rd, rs, cs, cl)
no_pole = serdes.ctle(f, gm, rd, rs, cs, 0.0)
i = with_pole.argmax()
dc = float(serdes.ctle(1.0, gm, rd, rs, cs, cl))
hf = float(serdes.ctle(1e15, gm, rd, rs, cs, 0.0))
print(f"model: DC {dc:.3f}, peak {with_pole[i]:.3f} at {f[i]/1e9:.2f} GHz, "
      f"at 16 GHz {serdes.ctle(16e9, gm, rd, rs, cs, cl):.4f}")
print(f"without the output pole the gain rises to {hf:.3f}: {20*np.log10(hf/dc):.2f} dB of peaking")
sim = ngspice("ctle.cir")
print(f"ngspice: DC {sim['gdc']:.3f}, peak {sim['gpk']:.3f} at {sim['fpk']/1e9:.2f} GHz, at 16 GHz {sim['g16']:.4f}")
plt.figure()
plt.semilogx(f / 1e9, 20 * np.log10(with_pole), label="with 40 fF load")
plt.semilogx(f / 1e9, 20 * np.log10(no_pole), "--", label="no output pole")
plt.xlabel("frequency (GHz)"); plt.ylabel("gain (dB)"); plt.legend(); plt.grid(True, which="both")
plt.show()

model: DC 0.800, peak 2.689 at 13.76 GHz, at 16 GHz 2.6638
without the output pole the gain rises to 4.000: 13.98 dB of peaking
ngspice: DC 0.800, peak 2.689 at 13.78 GHz, at 16 GHz 2.6638


## Part 2 · FFE against DFE

*Predict* the three zero-forcing taps for 24-02's channel and the equalized NRZ and PAM4 eyes.

In [3]:
h = {-1: 0.02156, 0: 0.73402, 1: 0.21088, 2: 0.02967, 3: 0.00345, 4: 0.00037}
c = serdes.zf_ffe(h, 1, 1)
g = serdes.convolve_cursors(c, h)
print("taps:", {k: round(float(x), 4) for k, x in c.items()})
print(f"unequalized: NRZ {serdes.eye(h):.3f}, PAM4 {serdes.eye(h, 4):.3f}")
print(f"TX FFE:      NRZ {serdes.eye(g):.3f}, PAM4 {serdes.eye(g, 4):.3f} (main cursor {g[0]:.4f})")
print(f"2-tap DFE:   NRZ {serdes.eye(h, skip=(0, 1, 2)):.3f}, PAM4 {serdes.eye(h, 4, skip=(0, 1, 2)):.3f}")

taps: {-1: -0.0223, 0: 0.7602, 1: -0.2175}
unequalized: NRZ 0.468, PAM4 -0.021
TX FFE:      NRZ 0.520, PAM4 0.155 (main cursor 0.5486)
2-tap DFE:   NRZ 0.709, PAM4 0.219


## Part 3 · DFE error propagation

$h_0$ = 1, $h_1$ = 0.5, noise set for a 10⁻³ error rate without propagation. *Predict* the fraction of errors that follow another error.

In [4]:
sigma = 1 / norm.isf(1e-3)
rate, cont = serdes.simulate_dfe(1.0, 0.5, sigma, 400_000)
print(f"sigma = {sigma:.3f}; simulated error rate {rate:.2e}; P(error | previous error) {cont:.3f} "
      f"(formula {serdes.p_continue(1.0, 0.5, sigma):.3f})")

sigma = 0.324; simulated error rate 1.42e-03; P(error | previous error) 0.256 (formula 0.250)


## Part 4 · Sign-sign LMS

*Predict* where a two-tap DFE and the data level settle on 24-02's channel.

In [5]:
taps, levs = serdes.lms_dfe(h, 0.02, 2, 2 ** -9, 40_000)
print(f"settled taps {taps[-5000:].mean(axis=0).round(3)}, data level {levs[-5000:].mean():.3f} "
      f"(targets {h[1]:.3f}, {h[2]:.3f}, {h[0]:.3f})")
plt.figure()
plt.plot(taps[:, 0], label="b1"); plt.plot(taps[:, 1], label="b2"); plt.plot(levs, label="data level")
for y in (h[1], h[2], h[0]):
    plt.axhline(y, color="0.6", ls=":")
plt.xlabel("symbol"); plt.ylabel("value"); plt.legend(); plt.grid(True)
plt.show()

settled taps [0.209 0.03 ], data level 0.735 (targets 0.211, 0.030, 0.734)


## Part 5 · Bang-bang CDR

Step $2^{-14}$ UI, 0.02 UI rms jitter, 100 ppm offset. *Predict* which loop keeps lock.

In [6]:
print(f"linearized bandwidth {serdes.bb_bandwidth(2**-14, 0.5, 0.02, 53.125e9)/1e6:.1f} MHz; "
      f"proportional slew limit {0.5 * 2**-14 * 1e6:.1f} ppm")
ep, _ = serdes.bb_cdr(100, 2 ** -14, 0.0, 0.02, 400_000)
epi, integ = serdes.bb_cdr(100, 2 ** -14, 2 ** -20, 0.02, 400_000)
print(f"proportional only: final error {ep[-1]:.1f} UI (cycle slips)")
print(f"with integral path: worst error after settling {np.abs(epi[200_000:]).max():.3f} UI, "
      f"integrator {integ[-50_000:].mean():.2e} UI/UI")
plt.figure()
plt.plot(ep, label="proportional only"); plt.plot(epi, label="proportional + integral")
plt.ylim(-1, 5); plt.xlabel("UI"); plt.ylabel("phase error (UI)"); plt.legend(); plt.grid(True)
plt.show()

linearized bandwidth 10.3 MHz; proportional slew limit 30.5 ppm


proportional only: final error 27.8 UI (cycle slips)
with integral path: worst error after settling 0.011 UI, integrator 9.98e-05 UI/UI
